In [1]:
import os
import json
from typing import TypedDict, Optional


In [2]:
!pip install -q langchain-google-genai langgraph



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
email_content = """
Subject: Flight Booking Request - Pune to Delhi

Dear Travel Team,

My name is Sandy Patil. I would like to book a flight from Pune to Delhi.

Travel Details:
- Departure Date: 20 July 2026
- Return Date: 25 July 2026
- Number of Passengers: 2
- Travel Class: Economy
- Preferred Airline: IndiGo (if available)
- Budget: ₹7,000 per passenger

Please also suggest the cheapest available flight.

Thank you.
"""


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# Make sure GOOGLE_API_KEY is set in your environment before running this.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


In [5]:
class TravelState(TypedDict):
    # Raw input
    email_content: str

    # Parsed trip request (filled in by parse_request_agent)
    source: str
    destination: str
    departure_date: str
    return_date: str
    passengers: int
    rooms: int
    preferred_airline: Optional[str]
    budget_per_passenger: Optional[float]

    # Flight
    flight_results: list
    best_flight: dict

    # Hotel
    hotel_results: list
    best_hotel: dict

    # Booking
    booking_confirmation: str

    # Trail of decisions / warnings surfaced to the user
    notes: list


In [7]:
def parse_request_agent(state: TravelState):
    print("[Request Parser] Reading customer email...")

    prompt = f"""
    Extract the following trip details from the email below and return ONLY
    valid JSON with these exact keys (no markdown fences, no commentary):

    - source (city name)
    - destination (city name)
    - departure_date (YYYY-MM-DD)
    - return_date (YYYY-MM-DD)
    - passengers (integer)
    - rooms (integer, guess 1 room per 2 passengers if not stated)
    - preferred_airline (string or null if none mentioned)
    - budget_per_passenger (number in INR, or null if not mentioned)

    Email:
    ---
    {state['email_content']}
    ---
    """

    notes = []
    try:
        response = llm.invoke(prompt)
        raw = response.content.strip()
        # Strip accidental markdown fences if the model adds them anyway
        if raw.startswith("```"):
            raw = raw.strip("`")
            raw = raw.split("\n", 1)[1] if "\n" in raw else raw
            raw = raw.replace("json", "", 1) if raw.lower().startswith("json") else raw
        parsed = json.loads(raw)
    except Exception as e:
        notes.append(f"Could not parse email with LLM ({e}); falling back to manual defaults.")
        parsed = {
            "source": "Pune",
            "destination": "Delhi",
            "departure_date": "2026-07-20",
            "return_date": "2026-07-25",
            "passengers": 2,
            "rooms": 1,
            "preferred_airline": "IndiGo",
            "budget_per_passenger": 7000,
        }

    print("Parsed request:", parsed)

    return {
        "source": parsed.get("source"),
        "destination": parsed.get("destination"),
        "departure_date": parsed.get("departure_date"),
        "return_date": parsed.get("return_date"),
        "passengers": parsed.get("passengers") or 1,
        "rooms": parsed.get("rooms") or 1,
        "preferred_airline": parsed.get("preferred_airline"),
        "budget_per_passenger": parsed.get("budget_per_passenger"),
        "notes": notes,
    }


In [8]:
def flight_search_agent(state: TravelState):
    print(f"[Flight Search] Searching flights {state['source']} -> {state['destination']} on {state['departure_date']}...")

    # TODO: replace with a real flight search API call
    flights = [
        {"airline": "IndiGo", "price": 5400, "duration": "2h 10m"},
        {"airline": "Air India", "price": 6200, "duration": "2h"},
        {"airline": "Akasa", "price": 5100, "duration": "2h 20m"},
    ]

    print("Flights found:", flights)
    return {"flight_results": flights}


In [9]:
def best_flight_agent(state: TravelState):
    print("[Flight Analyst] Selecting best flight...")

    flights = state["flight_results"]
    budget = state.get("budget_per_passenger")
    preferred = state.get("preferred_airline")
    notes = list(state.get("notes", []))

    if not flights:
        notes.append("No flights returned by search.")
        return {"best_flight": {}, "notes": notes}

    in_budget = [f for f in flights if budget is None or f["price"] <= budget]

    if not in_budget:
        best = min(flights, key=lambda x: x["price"])
        notes.append(
            f"No flight found within budget of Rs.{budget}/passenger; "
            f"showing cheapest available option (Rs.{best['price']}) instead."
        )
    else:
        preferred_matches = [f for f in in_budget if preferred and f["airline"].lower() == preferred.lower()]
        if preferred_matches:
            best = min(preferred_matches, key=lambda x: x["price"])
        else:
            best = min(in_budget, key=lambda x: x["price"])
            if preferred:
                notes.append(f"Preferred airline '{preferred}' not available in budget; picked cheapest in-budget flight instead.")

    print("Best flight:", best)
    return {"best_flight": best, "notes": notes}


In [10]:
def hotel_search_agent(state: TravelState):
    print("[Hotel Search] Searching hotels...")

    # TODO: replace with a real hotel search API call
    hotels = [
        {"hotel": "Taj Hotel", "price": 8500, "rating": 4.8},
        {"hotel": "Holiday Inn", "price": 6000, "rating": 4.5},
        {"hotel": "Lemon Tree", "price": 5500, "rating": 4.2},
    ]

    print("Hotels found:", hotels)
    return {"hotel_results": hotels}


In [11]:
def best_hotel_agent(state: TravelState):
    print("[Hotel Analyst] Selecting best hotel...")

    hotels = state["hotel_results"]
    notes = list(state.get("notes", []))

    if not hotels:
        notes.append("No hotels returned by search.")
        return {"best_hotel": {}, "notes": notes}

    # Score = rating per 1000 currency units spent; higher is better value.
    best = max(hotels, key=lambda h: h["rating"] / (h["price"] / 1000))

    print("Best hotel:", best)
    return {"best_hotel": best, "notes": notes}


In [12]:
def booking_manager_agent(state: TravelState):
    print("[Booking Manager] Preparing booking...")

    if not state.get("best_flight") or not state.get("best_hotel"):
        message = "Booking could not be completed: missing flight or hotel selection."
        print(message)
        return {"booking_confirmation": message}

    notes = state.get("notes", [])
    notes_block = ("\n\nNote(s):\n- " + "\n- ".join(notes)) if notes else ""

    prompt = f"""
    Passenger: from the original request
    Passengers: {state['passengers']}
    Rooms: {state['rooms']}

    Flight:
    {state['best_flight']}

    Hotel:
    {state['best_hotel']}

    Warnings to mention to the customer, if any:
    {notes if notes else "None"}

    Prepare a short, friendly booking confirmation. If there are warnings,
    mention them plainly (e.g. budget or airline preference not fully met).
    """

    try:
        response = llm.invoke(prompt)
        confirmation = response.content
    except Exception as e:
        confirmation = (
            f"Flight: {state['best_flight']}\nHotel: {state['best_hotel']}"
            f"{notes_block}\n\n(Could not generate LLM summary: {e})"
        )

    print("Booking confirmation:", confirmation)
    return {"booking_confirmation": confirmation}


In [13]:
workflow = StateGraph(TravelState)

workflow.add_node("parse_request", parse_request_agent)
workflow.add_node("flight_search", flight_search_agent)
workflow.add_node("best_flight", best_flight_agent)
workflow.add_node("hotel_search", hotel_search_agent)
workflow.add_node("best_hotel", best_hotel_agent)
workflow.add_node("booking_manager", booking_manager_agent)

workflow.add_edge(START, "parse_request")
workflow.add_edge("parse_request", "flight_search")
workflow.add_edge("flight_search", "best_flight")
workflow.add_edge("best_flight", "hotel_search")
workflow.add_edge("hotel_search", "best_hotel")
workflow.add_edge("best_hotel", "booking_manager")
workflow.add_edge("booking_manager", END)

app = workflow.compile()


In [14]:
initial_data = {
    "email_content": email_content,
    "flight_results": [],
    "best_flight": {},
    "hotel_results": [],
    "best_hotel": {},
    "booking_confirmation": "",
    "notes": [],
}

result = app.invoke(initial_data)

print("\n--- Final state ---")
print(result)


[Request Parser] Reading customer email...
Parsed request: {'source': 'Pune', 'destination': 'Delhi', 'departure_date': '2026-07-20', 'return_date': '2026-07-25', 'passengers': 2, 'rooms': 1, 'preferred_airline': 'IndiGo', 'budget_per_passenger': 7000}
[Flight Search] Searching flights Pune -> Delhi on 2026-07-20...
Flights found: [{'airline': 'IndiGo', 'price': 5400, 'duration': '2h 10m'}, {'airline': 'Air India', 'price': 6200, 'duration': '2h'}, {'airline': 'Akasa', 'price': 5100, 'duration': '2h 20m'}]
[Flight Analyst] Selecting best flight...
Best flight: {'airline': 'IndiGo', 'price': 5400, 'duration': '2h 10m'}
[Hotel Search] Searching hotels...
Hotels found: [{'hotel': 'Taj Hotel', 'price': 8500, 'rating': 4.8}, {'hotel': 'Holiday Inn', 'price': 6000, 'rating': 4.5}, {'hotel': 'Lemon Tree', 'price': 5500, 'rating': 4.2}]
[Hotel Analyst] Selecting best hotel...
Best hotel: {'hotel': 'Lemon Tree', 'price': 5500, 'rating': 4.2}
[Booking Manager] Preparing booking...
Booking confir